# Week 3 Day 5: AI Meeting Minutes Generator

Today we build a **real AI product** that takes a recording of a meeting and automatically generates structured meeting minutes.

This project chains **two AI models together** to go from raw audio → polished notes:

| Step | What happens | Model / Tool |
|------|-------------|-------------|
| 1 | **Transcribe** the audio into text | Whisper (open-source) OR OpenAI API |
| 2 | **Summarise** the transcript into meeting minutes | Llama 3.2 3B (open-source, 4-bit quantized) |

**Key Skills You'll Practice:**
- Using Hugging Face `pipeline()` for speech recognition (Whisper)
- Using the OpenAI API as an alternative transcription method
- Mounting Google Drive in Colab to load audio files
- Using the low-level Transformers API (`AutoTokenizer` + `AutoModelForCausalLM`)
- 4-bit quantization with `BitsAndBytesConfig`
- Streaming output with `TextStreamer`
- Prompt engineering to get structured Markdown output from an LLM

---
## Step 1 — Install Required Libraries

- **`transformers`** — Hugging Face library for models, tokenizers, and pipelines
- **`accelerate`** — Helps load large models efficiently across CPU/GPU
- **`bitsandbytes`** — Enables 4-bit quantization so big models fit on a free T4 GPU


In [1]:
!pip install -qU bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.7 MB/s eta 0:00:00


## Step 2 — Import Everything We Need

In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

## Step 3 — Define the Model

In [3]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"


## Step 4 — Connect to Google Drive

**New capability!** We connect this Colab notebook to your Google Drive so we can read the audio file directly.

**How it works:**
1. `drive.mount("/content/drive")` — makes your entire Google Drive accessible as a folder inside Colab
2. We then set `audio_filename` to the path where the `.mp3` file lives on your drive

> **Setup:** Download `denver_extract.mp3` from the link above and place it in a folder called `llms` on your Google Drive.


In [4]:
# New capability - connect this Colab to your Google Drive
# See immediately below this for instructions to obtain denver_extract.mp3
# Place the file on your drive in a folder called llms, and call it denver_extract.mp3

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/Colab Notebooks/LLM Week 3/denver_extract.mp3"


Mounted at /content/drive


## Step 5 — Log In to Hugging Face & Open the Audio File

We need Hugging Face authentication to download the Llama model (it requires access approval).

We also open the audio file in binary mode (`"rb"`) — this is required when sending files to the OpenAI API.


In [5]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the audio file in binary mode (needed for the OpenAI API option later)

audio_file = open(audio_filename, "rb")


---
# STEP 6: Transcribe Audio

Before we can generate meeting minutes, we need **text**. The audio is just sound waves — we need a model to convert speech into words.

We'll try **two approaches** and compare them:

| | Option 1: Open-Source (Whisper) | Option 2: OpenAI API |
|---|---|---|
| **Cost** | Free (uses our GPU) | Pay per request |
| **Privacy** | Audio stays on our machine | Audio sent to OpenAI's servers |
| **Model** | `openai/whisper-medium.en` | `gpt-4o-mini-transcribe` |
| **How** | Hugging Face `pipeline()` | OpenAI Python client |


## Option 1: Open-Source Transcription with Whisper (Hugging Face Pipeline)

**What is Whisper?**  
Whisper is an open-source speech recognition model released by OpenAI. It converts spoken audio into text. Despite being made by OpenAI, it's fully open-source and available on Hugging Face — we can run it locally for free!

**How it works step by step:**
1. We create a `pipeline("automatic-speech-recognition")` — this loads the Whisper model and all its audio preprocessing
2. We specify `model="openai/whisper-medium.en"` — the medium-sized English-only variant (good balance of speed and accuracy)
3. We pass `dtype=torch.float16` to load the model in half precision (uses less GPU memory)
4. We set `device='cuda'` to run on the GPU
5. We pass `return_timestamps=True` — this helps the model handle longer audio files correctly
6. We feed in the audio file path, and the pipeline returns a dictionary with the transcription in `result["text"]`


In [6]:
from transformers import pipeline

# Create a speech recognition pipeline using the open-source Whisper model
pipe = pipeline(
    "automatic-speech-recognition",       # The task: convert speech to text
    model="openai/whisper-medium.en",      # Whisper medium English — good balance of speed vs accuracy
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

# Run the pipeline on our audio file
result = pipe(audio_filename)

# Extract just the text from the result dictionary
transcription = result["text"]


config.json:   0%|          | 0.00/1.95k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.06GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/1.95k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to

In [7]:
transcription

" kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So, thank you. Thank you so much and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the

In [8]:
# Save the open-source transcription separately so we can compare it with the OpenAI version later
open_source_transcription = transcription


## Option 2: Groq API Transcription

As an alternative, we can use Groq's fast and cost-effective API for transcription.

**How it works:**
1. We create a compatibility client using our Groq API key (stored in Colab Secrets as `GROQ` or `GROQ_API_KEY`)
2. We configure the base URL to `https://api.groq.com/openai/v1`
3. We call `openai.audio.transcriptions.create()` with `whisper-large-v3-turbo`

This is a super fast and cloud-based alternative to running Whisper locally on your own GPU.

In [9]:
# Sign in to OpenAI using Secrets in Colab
AUDIO_MODEL = "whisper-large-v3-turbo"

groq_api_key = userdata.get('GROQ')

openai = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Send the audio file to OpenAI for transcription
transcription = openai.audio.transcriptions.create(
    model=AUDIO_MODEL,
    file=audio_file,
    response_format="text"
)


In [10]:
transcription

" kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so let me see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these Indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the

### Compare Both Transcriptions

Let's display both side by side so we can see how the open-source local Whisper model compares to the Groq cloud API.

In [11]:
# Display both transcriptions for comparison
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

 kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So, thank you. Thank you so much and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America, and to the republic for which it stands, one nation under God, indivisible, with liberty and justice for all. All right, thank you, Councilman Lopez, Madam Secretary, roll call. Black. Clark. Here. Espinosa. Here. Here Flynn Gilmore here Here cashman here can each Lopez yeah new Ortega here Sussman mr. President here 11 present 11 members present we do have a quorum approval the minutes are there any corrections to the minutes of October 2nd Seeing none minutes of October 2nd stand approve council announcements. Are there any announcements by members of council? Councilman Clark, thank you. Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway and lucky district 7 it will happen on Saturday, October 21st at 6 o'clock p.m It will move along Broadway from 3rd to Alameda It's gonna be a fun family-friendly event. Everyone's invited to come down wear a costume there will be candy for the kids and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on The fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween bread. Thank you Mr. President. All right. Thank you councilman Clark. I will be there. All right Presentations, Madam Secretary, do we have any presentations? None, Mr. President. Communications, do we have any communications? None, Mr. President. We do have one proclamation this evening, Proclamation 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President, with pride. proclamation number 17, well let me just say this differently, proclamation number 1127 series of 2017 in observance of the second annual Indigenous Peoples Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the indigenous peoples have lived and flourished on the lands known as the America since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous indigenous tribes, which include the Southern Ute, the Ute Mountain Ute tribes of Colorado, and whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes and the City and County of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas, on October 3rd, 2016, the City and County of Denver unanimously passed Council Bill 801, Series of 2016, officially designating the second Monday of October of each year as Indigenous Peoples Day in Denver, Colorado. And whereas, the Council of the City and County of Denver continues to recognize and value the vast contributions made to community made to the community through indigenous people's knowledge, science, philosophy, arts and culture and through these contributions the City of Denver has developed and thrived. Whereas the indigenous community especially you have made great efforts this year to draw attention to the contributions of indigenous people including Confluence once a week, drawing record attendance to a national Indigenous youth leadership conference, leading conversations on inclusion with their peers, and supporting increased Indigenous youth participation in science and engineering. Now therefore, be it proclaimed by the Council of the City and County of Denver, Section 1, that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history, our past, our present, future and continues to promote the education of the Denver community about these historical and contemporary contributions of indigenous people. Section 2, that the City and County of Denver, Colorado does hereby observe October 9, 2017 as Indigenous Peoples' Day. Section 3, that the Clerk of the City and County of Denver shall attest and affix the seal of the city and county of Denver to this proclamation and that a copy be transmitted to the Denver American Indian Commission, the City and County of Denver School District number one, and the Colorado Commission on Indian Affairs. Thank you, Councilman Lopez. Your motion to adopt? Mr. President, I move that proclamation number 1127 series of 2017 be adopted. All right, it has been moved and seconded. Council members of Thank you, Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for the third time, but as Indigenous Peoples Day in Denver officially for the second time. It's always awesome to be able to see not just this proclamation come by my desk, but to see so many different people from our community in our council chambers. It was a very beautiful piece of artwork that you presented to us earlier. And it is exactly the spirit that we drafted this proclamation. And the ordinance that created Indigenous Peoples' Day, when we sat down and wrote it as a community, we couldn't think of anything else to begin except for the confluence of the two rivers. And those confluence of the two rivers created such a great city. And we live in such an amazing city, and we're all proud of it. And sometimes we, and a lot of people from all over the country are all over the world are proud of it. And sometimes a little too proud of it to tell them to go back home. But I'm kidding when I say that. But the really nice thing about this is that we are celebrating Indigenous People's Day out of pride for who we are, who we are as a city and the contributions of indigenous people to the city, not out of spite, not out of a replacement of one culture over the other, or out of contempt or disrespect. I think of a quote that Cesar Chavez made very popular. And it stuck with me for a very long time. And any time I have the opportunity, I just speak in front of children and especially children in our community that, you know, they often second guess themselves and where they're coming from or who they are. And I always say that, you know, it's very important to be proud of who you're from. And the quote that I use from Cesar Chavez is, you know, pride in one's own culture does not require contempt or disrespect of another, right? And that's very important. It's very important for us to recognize that no matter who we are, where we come from in this society, that your pride in your own culture doesn't require, should not require the contempt or disrespect of another. And man, what a year for that to just sit on our shoulders for a while, for us to think about, right? And so I wanted to just to thank you all, I think the commission, there's going to be a couple of individuals that are going to come and speak thank you for your art, your lovely artwork for us to see what's in your heart and what now has become probably is going to be a very important symbol for the community. And also just for the work, the daily work, every single day. We still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners, right, in poverty without access to services, right? Without access to sobriety or even housing or jobs. And what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores, right? So we have a lot of work to do. And these kind of proclamations in this day is not a day off, it's a day on in Denver, right? and addressing those critical issues. So I know that my colleagues are very supportive. I'm going to ask you to support this proclamation. I know you always have done in the past. I'm very proud of today. Oh, and we made Time Magazine and Newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous Peoples Day. I wanted to make a point out of that. Thank you, Councilman Lopez, and thank you for sponsoring this. Councilman Martella. Mr. President, I want to ask that my name be added. I don't think I could add much more to what Councilman Lopez has shared with us. I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state. I worked in the Lieutenant Governor's Office when the Commission on Indian Affairs was created and had the benefit of being able to go down to the four corners for a peace treaty signing ceremony between the Utes and the Comanches that had been sort of at odds with each other for about 100 years and just being able to participate in that powwow was pretty awesome. So, and for those of you who continue to participate in the annual powwow, it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture. I mean, to see that the dance continues to be carried on as well as the native language from generation to generation is just so incredible because in so many cultures, people have come here and assimilated to the norms here you know the norms here and they lose their language and and and lose a lot of the culture and in the native community that that hasn't happened that has that you know commitment to just passing that on from generation to generation is is so important and so I'm happy to be a co-sponsor of this tonight thank you All right. Thank you, Councilwoman Ortega. Councilwoman Knitsche. Thank you very much, and I also want to thank my colleague for bringing this forward. And I just wanted to say a word to the artist about how beautiful and moving I thought this logo was and your description of it. And I think one of the things that is clear is, you know, the words sometimes don't convey the power of imagery or music or the other pieces that make up culture, and so I think the art is so important. And when you talked about water I was also thinking about land and I guess I just wanted to say thank you many of the Native American peoples of Colorado have been at the forefront or actually nationally of defending some of the the public lands that have been protected over the last few years that are under attack right now and they're places that you are the communities have fought to protect but that everyone gets to enjoy and so I just think that it's an example of where cultural preservation intersects with environmental protection with you know recreation and all of the other ways that that public lands are so important and so I think I just wanted to say thank you for that because I think we have some very sacred places in our country that are at risk right now and so as we celebrate I appreciate that there's still a piece of resistance in here and I think that I just want to mention a solidarity and I mentioned a feeling of solidarity with that resistance. So, thank you and happy Confluence Week. Thank you, Councilwoman Kniech. And seeing no other comments, I'll just say a couple. And in a time of such divisive ugliness and just despicable behavior from our leadership, the reason I'm so supportive of Indigenous Peoples' Day is because it means inclusivity. It means respecting all, respecting those who have been silenced on purpose for a long time and whose history has not been told. And so we celebrate inclusivity in the face of such evil times, honestly.

 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so let me see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these Indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America, to the republic for which One nation, under God, indivisible, with liberty and justice for all. All right, thank you, Councilman Lopez. Madam Secretary, roll call. Black? Clark? Here. Espinosa? Here. Flynn? Gilmore? Here. Here. Cashman? Here. Canich? Here. Lopez? Here. New? Here. Ortega? Here. Sussman? Here. Mr. President? Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family-friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids, and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween bread. Thank you, Mr. President. All right. Thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do we have any presentations? None. Mr. President? Communications. Do we have any communications? None. Mr. President? We do have one proclamation this evening, Proclamation 1127, 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Loplaz, will you please read it? Thank you, Mr. President, with pride. Proclamation number 17, well, let me just say this differently. Proclamation number 1127 series of 2017 an observance of the second annual Indigenous Peoples Day in the City and County of Denver Whereas the Council of the city and county of Denver recognizes that the Indigenous peoples have lived and flourished on the lands known as the America since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes, which include the Southern Ute, the Ute Mountain, Ute tribes of Colorado, and whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes, and the city and county of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. and whereas on October 3rd, 2016, the City and County of Denver unanimously passed Council Bill 801 Series of 2016 officially designating the second Monday of October of each year as Indigenous Peoples Day in Denver, Colorado and whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to the community through Indigenous peoples' knowledge, science, philosophy, arts, and culture, and through these contributions, the City of Denver has developed and thrived. Whereas the Indigenous community, especially youth, have made great efforts this year to draw attention to the contributions of Indigenous people, including Confluence Week, drawing record attendance to a national Indigenous youth leadership conference leading conversations on inclusion with their peers and supporting increased Indigenous youth participation in science and engineering. Now therefore, be it proclaimed by the Council of the City and County of Denver, Section 1, that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history, our past, our present, and future. and continues to promote the education of the Denver community about these historical and contemporary contributions of Indigenous people. Section 2, that the City and County of Denver, Colorado, does hereby observe October 9, 2017 as Indigenous Peoples Day. Section 3, that the Clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation, and that a copy be transmitted to the Denver American Indian Commission, the City and County of Denver School District No. 1, and the Colorado Commission on Indian Affairs. Thank you, Councilman Lopez. Your motion to adopt. Mr. President, I move that Proclamation No. 1127, Series of 2017, be adopted. All right. It has been moved and seconded. It comes by members of Council. Councilman Lopez. Thank you, Mr. President. it gives me a lot of pleasure and pride to read this proclamation officially for the third time, but as Indigenous Peoples Day in Denver officially for the second time. It's always awesome to be able to see not just this proclamation come by my desk, but to see so many different people from our community in our council chambers. it was a very beautiful piece of artwork that you presented to us earlier and it is exactly the spirit that we drafted this proclamation and this actual the ordinance that created indigenous people day when we sat down and wrote it and as a community we couldn't think of anything else to begin except for the confluence of the two rivers and those confluence of the two rivers created such a great city and we live in such an amazing city and we we're all proud of it and sometimes we um and a lot of people from all over the country or all over the world are proud of it and sometimes a little too proud of it just tell them to go back home um but um i'm kidding when i say that um but the the really nice thing about this is that um we are celebrating indigenous people's day out of pride for who we are, who we are as a city, and the contributions of indigenous people to the city, not out of spite, not out of a replacement of one culture over the other, or out of contempt or disrespect. You know, I think of a quote that Cesar Chavez made very popular, and it stuck with me for a very long time and anytime I have the opportunity to speak in front of children and especially children in our community that often second guess themselves on where they're coming from, who they are and I always say that it's very important to be proud of who you're from and the quote that I use from Sister Chavez is, you know, pride in one's own culture does not require contempt or disrespect of another, right? And that's very important. It's very important for us to recognize that no matter who we are, where we come from in this society, that your pride in your own culture doesn't require, should not require the contempt or disrespect of another. And man, what a year for that to just sit on our shoulders for a while, for us to think about, right? and so I wanted to just to thank you all, I think the commission there's going to be a couple individuals that are going to come speak, thank you for your art, your lovely artwork for us to see what's in your heart and what now has become probably going to be a very important symbol for the community and also just for the work the daily work, every single day we still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners, right, in poverty, without access to services, right, without access to sobriety or even housing or jobs. And what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores, right? So we have a lot of work to do. And these kind of proclamations and this day is not a day off, it's a day on in Denver, right, in addressing those critical issues. So I know that my colleagues are very supportive. I'm going to ask you to support this proclamation, as I know you always have done in the past. I'm very proud of today. Oh, and we made Time Magazine and Newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous Peoples Day. I wanted to make a point out of that. Thank you. Councilman Lopez and thank you for sponsoring this council Marcega. Mr. President I want to ask that my name be added. I don't think I could add much more to what Councilman Lopez has shared with us I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state. I worked in the Lieutenant Governor's Office when the Commission on Indian Affairs was created and had the benefit of being able to go down to the Four Corners for a peace treaty signing ceremony between the Utes and the Comanches that had been sort of at odds with each other for about 100 years and just being able to participate in that powwow was pretty awesome. For those of you who continue to participate in the annual powwow, it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture. I mean, to see that the dance continues to be carried on as well as the Native language from generation to generation is just so incredible because in so many cultures, you know, people have come here and assimilated to the, you know, the norms here and they lose their language and lose a lot of the culture. And in the Native community, that hasn't happened. That commitment to just passing that on from generation to generation is so important. And so I'm happy to be a co-sponsor of this tonight. Thank you. All right. Thank you, Councilwoman Ortega. Councilwoman Kenneach. Thank you very much. And I also want to thank my colleague for bringing this forward. and I just wanted to say a word to the artist about how beautiful and moving I thought this logo was and your description of it and I think one of the things that is clear is you know the words sometimes don't convey the power of imagery or music or the other pieces that make up culture and so I think the art is so important and when you talked about water I was also thinking about land and I guess I just wanted to say thank you many of the Native American and peoples of Colorado have been at the forefront, or actually nationally, of defending some of the public lands that have been protected over the last few years that are under attack right now. And they're places that the communities have fought to protect, but that everyone gets to enjoy. And so I just think that it's an example of where cultural preservation intersects with environmental protection, with recreation and all of the other ways that public lands are so important. And so I think I just wanted to say thank you for that, because I think we have some very sacred places in our country that are at risk right now. And so as we celebrate, I appreciate that there's still a piece of resistance in here. And I think that I just want to mention a solidarity and mention a feeling of solidarity with that resistance. So thank you and happy Confluence Week. Thank you, Councilwoman Kenneach. and seeing no other comments, I'll just say a couple. And in a time of such divisive ugliness and just despicable behavior from our leadership, the reason I'm so supportive of Indigenous Peoples Day is because it means inclusivity. It means respecting all, respecting those who have been silenced on purpose for a long time and whose history has not been told. And so we celebrate inclusivity in the face of such evil times, honestly.

---
# Analyse & Report — Generate Meeting Minutes

Now we have our transcript (raw text). The next step is to feed it to an LLM and ask it to produce structured meeting minutes.

We'll use the **low-level Transformers API** (same approach from Day 4) with Llama 3.2 to:
1. Set up a system prompt that tells the model what role it plays
2. Include the transcript in a user prompt
3. Load the model with 4-bit quantization
4. Generate meeting minutes with streaming output


## Step 7 - Craft the Prompts

**Prompt engineering** is critical here. We use two messages:

- **System message** → tells the model WHO it is and WHAT format to use
- **User message** → gives the actual transcript and specific output requirements

Notice we ask for specific sections: summary, discussion points, takeaways, and action items. This gives the LLM a clear structure to follow.


In [12]:
# System message: tells the LLM what role it plays and what format to use
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

# User message: provides the transcript and tells the LLM exactly what sections we want
user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

# Put them together in the standard chat messages format
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

## Step 8 — Configure 4-Bit Quantization

This is the same quantization config from Day 4:

- `load_in_4bit=True` → compress model weights from 16-bit to 4-bit (uses ~4× less memory)
- `bnb_4bit_use_double_quant=True` → quantizes the quantization constants too (saves a bit more memory)
- `bnb_4bit_compute_dtype=torch.bfloat16` → does the actual math in bfloat16 (good balance of speed and precision)
- `bnb_4bit_quant_type="nf4"` → uses NormalFloat4 quantization, which works especially well for LLMs


In [13]:
# 4-bit quantization config — same as Day 4
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Load weights in 4-bit
    bnb_4bit_use_double_quant=True,          # Extra compression on quantization constants
    bnb_4bit_compute_dtype=torch.bfloat16,   # Compute in bfloat16 for speed
    bnb_4bit_quant_type="nf4"                # NormalFloat4 — best quantization type for LLMs
)

## Step 9 — Load Tokenizer, Format Prompt, Load Model & Generate

This follows the exact same pattern we learned in Day 4:

1. **Load the tokenizer** → `AutoTokenizer.from_pretrained(LLAMA)`
2. **Set the pad token** → Llama has no pad token by default, so we reuse the end-of-sequence token
3. **Apply the chat template** → converts our `messages` list into Llama's expected format with special tokens
4. **Create a TextStreamer** → so we can see the output appear word by word
5. **Load the model** → with 4-bit quantization and automatic device mapping
6. **Generate!** → `model.generate()` with `max_new_tokens=2000` because meeting minutes can be long

**Why `max_new_tokens=2000`?**  
In Day 4 we used 80 tokens for a joke. Meeting minutes are much longer — we need room for the summary, all discussion points, takeaways, and action items.

In [16]:
# 1. Load the tokenizer for Llama 3.2
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token   # Llama has no pad token — reuse end-of-sequence token

# 2. Convert our chat messages into Llama's expected format and into a GPU tensor
# Added return_dict=True and passed via unpacking to prevent AttributeError
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",    # Return as PyTorch tensors
    return_dict=True
).to("cuda")              # Move all input tensors to GPU

# 3. Create a streamer so we see output in real time (like ChatGPT)
streamer = TextStreamer(tokenizer)

# 4. Load the model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",                  # Automatically place layers on GPU/CPU
    quantization_config=quant_config     # Apply our 4-bit quantization
)

# 5. Generate meeting minutes! max_new_tokens=2000 because minutes are much longer than a joke
# We unpack inputs using ** to pass both input_ids and attention_mask correctly
outputs = model.generate(
    **inputs,
    max_new_tokens=2000,    # Meeting minutes need many tokens
    streamer=streamer       # Stream output word by word
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 06 Oct 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so let me see the creation of the logo here. And yeah, so that basically kind of sums up

## Step 10 — Decode the Output

`model.generate()` returns raw **token IDs** — just numbers.  
`tokenizer.decode()` converts those numbers back into readable text.

In [17]:
# Convert the generated token IDs back into readable text
response = tokenizer.decode(outputs[0])


## Step 11 — Display the Final Meeting Minutes

We asked the model to output in Markdown format. `display(Markdown(...))` renders it with proper headings, bullet points, and formatting — just like a real document!

In [18]:
# Render the meeting minutes as beautiful formatted Markdown
display(Markdown(response))


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 06 Oct 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
 kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so let me see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these Indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America, to the republic for which One nation, under God, indivisible, with liberty and justice for all. All right, thank you, Councilman Lopez. Madam Secretary, roll call. Black? Clark? Here. Espinosa? Here. Flynn? Gilmore? Here. Here. Cashman? Here. Canich? Here. Lopez? Here. New? Here. Ortega? Here. Sussman? Here. Mr. President? Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family-friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids, and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween bread. Thank you, Mr. President. All right. Thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do we have any presentations? None. Mr. President? Communications. Do we have any communications? None. Mr. President? We do have one proclamation this evening, Proclamation 1127, 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Loplaz, will you please read it? Thank you, Mr. President, with pride. Proclamation number 17, well, let me just say this differently. Proclamation number 1127 series of 2017 an observance of the second annual Indigenous Peoples Day in the City and County of Denver Whereas the Council of the city and county of Denver recognizes that the Indigenous peoples have lived and flourished on the lands known as the America since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes, which include the Southern Ute, the Ute Mountain, Ute tribes of Colorado, and whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes, and the city and county of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. and whereas on October 3rd, 2016, the City and County of Denver unanimously passed Council Bill 801 Series of 2016 officially designating the second Monday of October of each year as Indigenous Peoples Day in Denver, Colorado and whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to the community through Indigenous peoples' knowledge, science, philosophy, arts, and culture, and through these contributions, the City of Denver has developed and thrived. Whereas the Indigenous community, especially youth, have made great efforts this year to draw attention to the contributions of Indigenous people, including Confluence Week, drawing record attendance to a national Indigenous youth leadership conference leading conversations on inclusion with their peers and supporting increased Indigenous youth participation in science and engineering. Now therefore, be it proclaimed by the Council of the City and County of Denver, Section 1, that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history, our past, our present, and future. and continues to promote the education of the Denver community about these historical and contemporary contributions of Indigenous people. Section 2, that the City and County of Denver, Colorado, does hereby observe October 9, 2017 as Indigenous Peoples Day. Section 3, that the Clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation, and that a copy be transmitted to the Denver American Indian Commission, the City and County of Denver School District No. 1, and the Colorado Commission on Indian Affairs. Thank you, Councilman Lopez. Your motion to adopt. Mr. President, I move that Proclamation No. 1127, Series of 2017, be adopted. All right. It has been moved and seconded. It comes by members of Council. Councilman Lopez. Thank you, Mr. President. it gives me a lot of pleasure and pride to read this proclamation officially for the third time, but as Indigenous Peoples Day in Denver officially for the second time. It's always awesome to be able to see not just this proclamation come by my desk, but to see so many different people from our community in our council chambers. it was a very beautiful piece of artwork that you presented to us earlier and it is exactly the spirit that we drafted this proclamation and this actual the ordinance that created indigenous people day when we sat down and wrote it and as a community we couldn't think of anything else to begin except for the confluence of the two rivers and those confluence of the two rivers created such a great city and we live in such an amazing city and we we're all proud of it and sometimes we um and a lot of people from all over the country or all over the world are proud of it and sometimes a little too proud of it just tell them to go back home um but um i'm kidding when i say that um but the the really nice thing about this is that um we are celebrating indigenous people's day out of pride for who we are, who we are as a city, and the contributions of indigenous people to the city, not out of spite, not out of a replacement of one culture over the other, or out of contempt or disrespect. You know, I think of a quote that Cesar Chavez made very popular, and it stuck with me for a very long time and anytime I have the opportunity to speak in front of children and especially children in our community that often second guess themselves on where they're coming from, who they are and I always say that it's very important to be proud of who you're from and the quote that I use from Sister Chavez is, you know, pride in one's own culture does not require contempt or disrespect of another, right? And that's very important. It's very important for us to recognize that no matter who we are, where we come from in this society, that your pride in your own culture doesn't require, should not require the contempt or disrespect of another. And man, what a year for that to just sit on our shoulders for a while, for us to think about, right? and so I wanted to just to thank you all, I think the commission there's going to be a couple individuals that are going to come speak, thank you for your art, your lovely artwork for us to see what's in your heart and what now has become probably going to be a very important symbol for the community and also just for the work the daily work, every single day we still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners, right, in poverty, without access to services, right, without access to sobriety or even housing or jobs. And what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores, right? So we have a lot of work to do. And these kind of proclamations and this day is not a day off, it's a day on in Denver, right, in addressing those critical issues. So I know that my colleagues are very supportive. I'm going to ask you to support this proclamation, as I know you always have done in the past. I'm very proud of today. Oh, and we made Time Magazine and Newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous Peoples Day. I wanted to make a point out of that. Thank you. Councilman Lopez and thank you for sponsoring this council Marcega. Mr. President I want to ask that my name be added. I don't think I could add much more to what Councilman Lopez has shared with us I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state. I worked in the Lieutenant Governor's Office when the Commission on Indian Affairs was created and had the benefit of being able to go down to the Four Corners for a peace treaty signing ceremony between the Utes and the Comanches that had been sort of at odds with each other for about 100 years and just being able to participate in that powwow was pretty awesome. For those of you who continue to participate in the annual powwow, it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture. I mean, to see that the dance continues to be carried on as well as the Native language from generation to generation is just so incredible because in so many cultures, you know, people have come here and assimilated to the, you know, the norms here and they lose their language and lose a lot of the culture. And in the Native community, that hasn't happened. That commitment to just passing that on from generation to generation is so important. And so I'm happy to be a co-sponsor of this tonight. Thank you. All right. Thank you, Councilwoman Ortega. Councilwoman Kenneach. Thank you very much. And I also want to thank my colleague for bringing this forward. and I just wanted to say a word to the artist about how beautiful and moving I thought this logo was and your description of it and I think one of the things that is clear is you know the words sometimes don't convey the power of imagery or music or the other pieces that make up culture and so I think the art is so important and when you talked about water I was also thinking about land and I guess I just wanted to say thank you many of the Native American and peoples of Colorado have been at the forefront, or actually nationally, of defending some of the public lands that have been protected over the last few years that are under attack right now. And they're places that the communities have fought to protect, but that everyone gets to enjoy. And so I just think that it's an example of where cultural preservation intersects with environmental protection, with recreation and all of the other ways that public lands are so important. And so I think I just wanted to say thank you for that, because I think we have some very sacred places in our country that are at risk right now. And so as we celebrate, I appreciate that there's still a piece of resistance in here. And I think that I just want to mention a solidarity and mention a feeling of solidarity with that resistance. So thank you and happy Confluence Week. Thank you, Councilwoman Kenneach. and seeing no other comments, I'll just say a couple. And in a time of such divisive ugliness and just despicable behavior from our leadership, the reason I'm so supportive of Indigenous Peoples Day is because it means inclusivity. It means respecting all, respecting those who have been silenced on purpose for a long time and whose history has not been told. And so we celebrate inclusivity in the face of such evil times, honestly.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Denver City Council Meeting Minutes**
**Date:** Monday, October 9th
**Location:** Denver City Council Chambers
**Attendees:**
- Council Members: Black, Clark, Espinosa, Flynn, Gilmore, Cashman, Canich, Lopez, New, Ortega, Sussman
- Mayor: Mr. President
- Guests: Native American Community Members, Artists, and Representatives

**Summary:**
The Denver City Council met to discuss and adopt a proclamation honoring Indigenous Peoples Day. The proclamation was read by Councilman Lopez, and it recognized the contributions of Indigenous people to Denver's history, culture, and present. The council members expressed their support and appreciation for the Native American community's contributions to the city.

**Discussion Points:**

- The importance of recognizing and honoring Indigenous Peoples Day
- The contributions of Indigenous people to Denver's history, culture, and present
- The need for inclusivity and respect for all communities, especially those who have been silenced or marginalized
- The significance of public lands and the importance of protecting them for future generations
- The role of cultural preservation in environmental protection and recreation

**Takeaways:**

- The Denver City Council is committed to recognizing and honoring Indigenous Peoples Day
- The council members expressed their support and appreciation for the Native American community's contributions to the city
- The proclamation will be celebrated as a symbol of inclusivity and respect for all communities

**Action Items:**

- Councilman Lopez will continue to work on promoting Indigenous Peoples Day and recognizing the contributions of Indigenous people to Denver's history and culture
- Councilwoman Ortega will continue to support and co-sponsor the proclamation
- The council will consider future initiatives to promote inclusivity and respect for all communities

**Proclamation 1127, Series of 2017:**
The council adopted Proclamation 1127, Series of 2017, which recognized Indigenous Peoples Day in Denver. The proclamation was read by Councilman Lopez, and it acknowledged the contributions of Indigenous people to Denver's history, culture, and present.<|eot_id|>

---
## What We Built Today — The Complete Pipeline

```
Audio File (.mp3) on Google Drive
        │
        ▼
  ┌─────┴──────┐
  │            │
Whisper     Groq API
(free,      (cloud,
 local)      fast)
  │            │
  └─────┬──────┘
        │
        ▼
  Raw Text Transcript
        │
        ▼
  System + User Prompt
        │
        ▼
  AutoTokenizer → apply_chat_template()
        │
        ▼
  Llama 3.2 3B (4-bit quantized on T4 GPU)
        │
        ▼
  model.generate(max_new_tokens=2000)
        │
        ▼
  tokenizer.decode()
        │
        ▼
  Formatted Meeting Minutes (Markdown)
```